# Bước 11 – "10% đầu mỗi đợt" là đầu theo thời gian, hay chỉ là đầu theo thứ tự dòng trong file?

Lý do: ở bước 10, mạng CIC-ToN-IoT có 88–94% luồng trùng thời điểm bắt đầu với luồng khác. Khi trùng thời điểm, thứ tự trước/sau giữa các luồng là thứ tự dòng trong file gốc, không phải thời gian. Bước 11 kiểm tra xem cú tụt từ 0,997 xuống 0,71 ở CIC-ToN-IoT có phải do chuyện này không.

| Ô | Việc | Thời gian |
|---|---|---|
| 1 | Kiểm tra môi trường và mã nguồn | 1 phút |
| 2 | Kiểm tra cột thời gian và thử 3 cách chọn 10% trên cả 4 mạng | 5–15 phút |

**Không cần GPU (card đồ họa).** Kết quả lưu vào `MyDrive/TADP_Q1/results/b11_thu_tu/`.

## Ô 1 – Kiểm tra môi trường và mã nguồn

In [1]:
import os, sys, subprocess
from google.colab import drive
drive.mount("/content/drive")
CODE_DIR = "/content/drive/MyDrive/TADP_Q1/code"
sys.path.insert(0, CODE_DIR)
for m in [k for k in sys.modules if k.startswith("tadp")]:
    del sys.modules[m]
try:
    import lightgbm, pyarrow
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "lightgbm", "pyarrow"], check=True)
EXPECTED = {"__init__.py": 125, "cic_data.py": 8579, "config.py": 3263, "data.py": 9956, "diag_order.py": 7059,
            "experts.py": 5934, "metrics.py": 4002, "preprocess.py": 3601}
ok = True
for f, size in EXPECTED.items():
    p = os.path.join(CODE_DIR, "tadp", f)
    real = os.path.getsize(p) if os.path.exists(p) else None
    if real != size:
        ok = False
        print(f"   SAI: {f} – mong đợi {size} byte, thực tế {real}")
print("Mã nguồn:", "ĐẦY ĐỦ, đúng phiên bản bước 11" if ok else "CÓ LỖI – báo lại cho Claude")

Mounted at /content/drive
Mã nguồn: ĐẦY ĐỦ, đúng phiên bản bước 11


## Ô 2 – Chạy kiểm tra

Với mỗi mạng, in 3 dòng (3 cách chọn 10% cho phần thích nghi) rồi dòng "[tên mạng] xong":
- `dau`: 10% đầu mỗi đợt, luồng trùng thời điểm giữ thứ tự dòng trong file (cách đang dùng);
- `dau_pha_trung`: 10% đầu theo thời gian, nhưng luồng trùng thời điểm được xáo trộn ngẫu nhiên;
- `ngau_nhien`: 10% bốc ngẫu nhiên trong mỗi đợt.

Cuối cùng in Bảng 11.1 (cột thời gian), 11.2 (mười đợt lớn nhất mỗi mạng), 11.3 (so sánh 3 cách). Chạy xong báo Claude.

In [2]:
import sys, warnings
warnings.filterwarnings("ignore", message="X does not have valid feature names")
from google.colab import drive
drive.mount("/content/drive")
sys.path.insert(0, "/content/drive/MyDrive/TADP_Q1/code")
for m in [k for k in sys.modules if k.startswith("tadp")]:
    del sys.modules[m]
from tadp.config import get_config
from tadp.data import load_all, common_features
from tadp.diag_order import run_diag_order
B = "/content/drive/MyDrive/TADP_Q1"
DOMAINS_CIC = {"CIC-IDS2017": ["2017"], "CSE-CIC-IDS2018": ["2018"], "CIC-ToN-IoT": ["ton"], "CIC-UNSW-NB15": ["unsw"]}
cfg = get_config(RAW_DIR=f"{B}/raw_cic", CACHE_DIR=f"{B}/cache_cic", RESULT_DIR=f"{B}/results/b11_thu_tu",
                 DOMAINS=DOMAINS_CIC)
data = load_all(cfg)
cols = common_features(data, "no_id")
run_diag_order(cfg, data, cols, pv="P1_log")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
CIC-IDS2017: 999,712 dòng, 87 cột, tỷ lệ tấn công = 0.2411
CSE-CIC-IDS2018: 999,829 dòng, 87 cột, tỷ lệ tấn công = 0.0559
CIC-ToN-IoT: 1,000,028 dòng, 82 cột, tỷ lệ tấn công = 0.5302
CIC-UNSW-NB15: 1,000,399 dòng, 82 cột, tỷ lệ tấn công = 0.0251
   [CIC-IDS2017] dau           : Macro-F1 eval = 0.993
   [CIC-IDS2017] dau_pha_trung : Macro-F1 eval = 0.993
   [CIC-IDS2017] ngau_nhien    : Macro-F1 eval = 0.999
[CIC-IDS2017] xong trong 44 giây
   [CSE-CIC-IDS2018] dau           : Macro-F1 eval = 0.998
   [CSE-CIC-IDS2018] dau_pha_trung : Macro-F1 eval = 0.998
   [CSE-CIC-IDS2018] ngau_nhien    : Macro-F1 eval = 0.999
[CSE-CIC-IDS2018] xong trong 39 giây
   [CIC-ToN-IoT] dau           : Macro-F1 eval = 0.709
   [CIC-ToN-IoT] dau_pha_trung : Macro-F1 eval = 0.740
   [CIC-ToN-IoT] ngau_nhien    : Macro-F1 eval = 0.992
[CIC-ToN-IoT] xong trong 42 giây
   [CIC-UNSW-NB

(              mang  so_dong  so_moc_thoi_gian_khac_nhau  ty_le_moc_tron_giay  ty_le_moc_tron_phut  tan_cong_ty_le_thoi_gian_cua_10pc_dau_tb_theo_so_luong  tan_cong_so_dot_thoi_luong_bang_0  ty_le_luong_10pc_dau_trung_moc_voi_90pc_sau
 0      CIC-IDS2017   999712                      655973             0.000929             0.000016                                           0.347430                                       0                                      0.00007
 1  CSE-CIC-IDS2018   999829                      994436             0.000979             0.000023                                           0.094113                                       0                                      0.00000
 2      CIC-ToN-IoT  1000028                       67883             1.000000             0.016251                                           0.114572                                       0                                      0.00107
 3    CIC-UNSW-NB15  1000399                       87172    